# Лабораторная работа 1. Что сохраняет представление текста?

## 1. Первичный анализ данных

In [1]:
import pandas as pd

dataset = pd.read_csv("women-clothing-accessories.3-class.balanced.csv", sep = "\t")
print(dataset.info())


length_ds = pd.DataFrame(columns = ['char_length', 'word_length'])
length_ds['char_length'] = dataset['review'].astype(str).apply(len)
length_ds['word_length'] = dataset['review'].astype(str).apply(lambda x: len(x.split()))

counts = dataset.sentiment.value_counts();

print(f"Размер:  {dataset.size}")
print("\nКлассы:")
print(counts)
print("\nХарактеристики по длине текстов:")
print(length_ds.describe())
print("\nПримеры каждого класса:")
print(dataset[dataset['sentiment'] == 'negative'].sample(random_state=42).values)
print(dataset[dataset['sentiment'] == 'neautral'].sample(random_state=42).values)
print(dataset[dataset['sentiment'] == 'positive'].sample(random_state=42).values)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 90000 entries, 0 to 89999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     90000 non-null  object
 1   sentiment  90000 non-null  object
dtypes: object(2)
memory usage: 1.4+ MB
None
Размер:  180000

Классы:
sentiment
negative    30000
neautral    30000
positive    30000
Name: count, dtype: int64

Характеристики по длине текстов:
        char_length   word_length
count  90000.000000  90000.000000
mean     130.069311     20.011144
std      126.137185     19.885001
min        1.000000      0.000000
25%       51.000000      8.000000
50%       91.000000     14.000000
75%      164.000000     25.000000
max     1007.000000    193.000000

Примеры каждого класса:
[['заказала xl а пришло самый маленький размер' 'negative']]
[['Есть зацепки,тонкий Ден.На рост 175 подошло.Одеть 1раз можно..'
  'neautral']]
[['цена-качество,нормальная,села в облипку,узкая по плечам и груди,н

### Неоднозначные метки: 

1. "долго  шла  11 11 17 закази  сегодня  10 01 18"    negative - Такой отзыв мог бы быть и в категории neutral, так как негативной оценки о самом товаре нет
2. "не понравилось. Мне огромное. Ткань не плотная. дешевая."	neautral - Я бы класифицировала это как негативный отзыв. 
3.  "рубашка качественная, красивая, но, мой размер 44, взяла L, тесновато в руках (руки худые) в общем впритыкан, берите XL, рост 167 рукава только только, если поднять руки  то задирается, в бедрах отлично, нигде не давит, все свободно." positive - Есть как позитивные так и негативные комментарии о товаре. Больше похоже на нейтральный. Скорее всего в категории positive, так как оценки на 4 и 5 звёзд в датасете совместили в одну категорию 

## 2. Разбиение данных

In [2]:
from sklearn.model_selection import train_test_split

X = dataset['review']
Y = dataset['sentiment']

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.3, random_state=42, stratify=Y)

print("Обучающая выборка:", Y_train.value_counts());
print("\nТестовая:", Y_test.value_counts());


Обучающая выборка: sentiment
positive    21000
neautral    21000
negative    21000
Name: count, dtype: int64

Тестовая: sentiment
positive    9000
negative    9000
neautral    9000
Name: count, dtype: int64


## 3. Сравнение разреженных представлений

In [3]:
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

#функция для подсчёта и вывода результатов
def res(train_matrix, pred):
    
    #Вычисление характеристик матрицы
    shape = train_matrix.shape
    cells_cnt = shape[0] * shape[1]
    nnz_cells = train_matrix.nnz;
    voc_size = len(bow_vectorizer.vocabulary_)
    nnz_percent = round((nnz_cells/cells_cnt) * 100, 3)

    #Получение предсказаний
    acc = round(accuracy_score(Y_test, pred), 3)
    f1 =  round(f1_score(Y_test, pred, average='macro'), 3)
    
    print("Размер словаря:", voc_size)
    print("Размерность матрицы признаков:", shape)
    print("Доля ненулевых элементов в ней:", nnz_percent)
    print("Accuracy:", acc)
    print("Macro-F1:", f1)

clf = LinearSVC(random_state=42)

#Представление BoW
bow_vectorizer = CountVectorizer()
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

clf.fit(X_train_bow, Y_train)
pred_BOW = clf.predict(X_test_bow)
    
print("BoW")
res(X_train_bow, pred_BOW)

#представление Tf-Idf
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

clf.fit(X_train_tfidf, Y_train)
pred_TDIDF = clf.predict(X_test_tfidf)

print("\nTF-IDF")
res(X_train_tfidf, pred_TDIDF)

#ПРедставление Tf-Idf + unigrams + bigrams
tfidf_vectorizer_ub = TfidfVectorizer(ngram_range=(1, 2))
X_train_tfidf = tfidf_vectorizer_ub.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer_ub.transform(X_test)

clf.fit(X_train_tfidf, Y_train)
pred_TDIDF_ub = clf.predict(X_test_tfidf)

print("\nTF-IDF + unigrams + bigrams")
res(X_train_tfidf, pred_TDIDF_ub)


BoW
Размер словаря: 44944
Размерность матрицы признаков: (63000, 44944)
Доля ненулевых элементов в ней: 0.039
Accuracy: 0.71
Macro-F1: 0.709

TF-IDF
Размер словаря: 44944
Размерность матрицы признаков: (63000, 44944)
Доля ненулевых элементов в ней: 0.039
Accuracy: 0.721
Macro-F1: 0.721

TF-IDF + unigrams + bigrams
Размер словаря: 44944
Размерность матрицы признаков: (63000, 464213)
Доля ненулевых элементов в ней: 0.008
Accuracy: 0.735
Macro-F1: 0.734


| Представление               | Размер словаря | Размерность train-матрицы | Доля ненулевых элементов | Accuracy  | Macro-F1 |
|-----------------------------|----------------|---------------------------|--------------------------|-----------|----------|
| Count, unigrams             | 44944          | (63000, 44944)            | 0.039                    | 0.71      | 0.709    |
| TF-IDF, unigrams            | 44944          | (63000, 44944)            | 0.039                    | 0.721     | 0.721    |
| TF-IDF, unigrams + bigrams  | 44944          | (63000, 464213)           | 0.008                    | 0.735     | 0.734    |


## 4. Анализ ошибок модели

In [ ]:
#Найдите:
#- не менее 5 примеров, которые модель с TF-IDF + unigrams 
# классифицировала неправильно, а
# модель с TF-IDF + unigrams + bigrams — правильно;
#- не менее 5 примеров с обратным поведением;
#- не менее 5 примеров, на которых обе модели ошибаются

#функция для вывода результатов
def print_res(target):
    for ind in target:
        print("\nТекст:", X_test.loc[ind])
        print("Истинная метка:", Y_test.loc[ind])
        print("Метка обычного TF-IDF:", pred_TDIDF[Y_test.index.get_loc(ind)])
        print("Метка TF-IDF с unigrams + bigrams:", pred_TDIDF_ub[Y_test.index.get_loc(ind)])
    print('\n' + '-'*20)

mask1 = (Y_test != pred_TDIDF) & (Y_test == pred_TDIDF_ub)
inds1 = Y_test.index[mask1][:5]

mask2 = (Y_test != pred_TDIDF_ub) & (Y_test == pred_TDIDF)
inds2 = Y_test.index[mask2][:5]

mask3 = (Y_test != pred_TDIDF) & (Y_test != pred_TDIDF_ub)
inds3 = Y_test.index[mask3][:5]


print("Примеры, где TF-IDF + unigrams + bigrams справилась, а обычный TF-IDF нет:")
print_res(inds1)
print("\nПримеры с обратной ситуацией:")
print_res(inds2)
print("\nПримеры, где обе модели ошиблись:")
print_res(inds3)


Примеры, где TF-IDF + unigrams + bigrams справилась, а обычный TF-IDF нет:

Текст: Отправка была 2 недели,только я начала радоваться,что товар наконец-таки отправили,как через 3 дня статус обновился об отмене отгрузки.Я не стала больше ждать и надеяться -открыла спор и на другой день уже вернули деньги.
Истинная метка: negative
Метка обычного TF-IDF: neautral
Метка TF-IDF с unigrams + bigrams: negative

Текст: Качество классное. Но вот размер ппц мягко говоря. Заказала S но он какой-то не тот, горло на вверху и плечи для М сделано думаю.
Истинная метка: neautral
Метка обычного TF-IDF: negative
Метка TF-IDF с unigrams + bigrams: neautral

Текст: на рубашке брак ткани
Истинная метка: negative
Метка обычного TF-IDF: neautral
Метка TF-IDF с unigrams + bigrams: negative

Текст: брак, дыра, шов не прошит, деньги не вернули, предложили зашить самим
Истинная метка: negative
Метка обычного TF-IDF: neautral
Метка TF-IDF с unigrams + bigrams: negative

Текст: на 48 раз. XL маленькая очень,цвет не

Анализ примеров из 1 группы:

1. Текст: на 48 раз. XL маленькая очень,цвет не беж а персик или даже шампань,нитки тоненькие вязка нежная,неаккуратная,в целом приличный кардиганчик,спасибо,аа много спущенных петель,за это снимаю звезду

Истинная метка: neautral

Метка обычного TF-IDF: positive

Метка TF-IDF с unigrams + bigrams: neautral

Скорее всег, TF-IDF отнёс в категорию positive из-за слов "нежная", "приличный", "спасибо", которые имеют более позитивный окрас и чаще встречаються в категории positive. А вот выражения "нитки тоненькие", "спущенных петель" и "снимаю звезду", которые чётко указывают на негативный окраc, имеют смысл только для подхода с биграммами. 

2. Текст: брак, дыра, шов не прошит, деньги не вернули, предложили зашить самим

Истинная метка: negative

Метка обычного TF-IDF: neautral

Метка TF-IDF с unigrams + bigrams: negative

Логика та же. Для обычного TF-IDF словосочетания "не прошит", "не вернули" слова "прошит" и "вернули" достаточно нейтральные и могли встречаться в разных классах, в то время как для биграм они будут более негативными. 

Анализ примеров из 2 группы:

1. Текст: Цвет тёмно синий , не как на фото ! Ткань плотная 600 грамм . На  изнаночной стороне выточка прозжёная  дыра !  Не открыла спор , пожелела продавца  ! Размер 48-50 . Доставка  4 недели . Ткань плотная синтетика .

Истинная метка: neautral

Метка обычного TF-IDF: neautral

Метка TF-IDF с unigrams + bigrams: positive

Из позитивного здесь вижу только "ткань плотная", так что не могу с уверенностью сказать, почему в TF-IDF с биграммами отзыв получил метку positive. Возможно, её "позитивный" окраз перевешивает остальные "негативно" окрашенные слова. Либо нейтральные описания товара из этого текста могли чаще встречаться в категории positive и поэтому оцениваться как более позитивные, а слова наподобии "дыра" и "спор" из-за большего кол-ва информации получили меньший вес. 

2. Текст: качество ужасное и не совсем соответствует описанию.

Истинная метка: negative

Метка обычного TF-IDF: negative

Метка TF-IDF с unigrams + bigrams: neautral

Здесь TF-IDF выставил negative на основе слова "ужасное", а вот TF-IDF с биграммами могло ввести в заблуждение выражение "соответсвует описанию".

Анализ примеров из 3 группы:

1. Текст: товар пришел с браком!!! дырка на товаре!!!

Истинная метка: negative

Метка обычного TF-IDF: neautral

Метка TF-IDF с unigrams + bigrams: neautral

Опять не вполне понятно, почему такие метки. Но для TF-IDF уже были отзывы из первой группы со словами "дыра" и "брак", которые оценивались как нейтральные (с биграммами эти же отзывы значились как негативные). Для биграмм, напрмер, такие словосочетания (именно в таком порядке и формах) могли отуствовать в словаре и просто не засчитаться, либо они не вносят такой большой вес в предложения. Если эти слова встречались не только в классе negative, то они не будут таким чёткими маркерами этого класса.  

2. Текст: мне не понравилось! штаны короткие. кофта
 растянута как-будто. ткань синтетика. вообщем

Истинная метка: negative

Метка обычного TF-IDF: neautral

Метка TF-IDF с unigrams + bigrams: neautral

У обоих методов одинаковый результат, а значит биграммы здесь не сильно изменили ситуацию. По сути только "не понрвилось" однозначно негативное, а остальные фразы более нейтральные, так что они могли просто "перевесть". 


## 5. Что использует линейная модель?


In [ ]:
#Для каждого класса выведите признаки с наиболее характерными весами.
#Выберите несколько содержательных примеров признаков и проанализируйте их
import numpy as np

def Top_words(features, clf):
    coef = clf.coef_
    classes = clf.classes_

    for i in range(len(classes)):
        class_coef = coef[i]
        top_inds = np.argsort(class_coef)[-15:][::-1]

        print("\nНаиболее характерные признаки для класса", classes[i])
        for ind in top_inds:
            print(features[ind], round(class_coef[ind], 5))
        
        print("\n" + "-"*20)
    
print("Оценка для TF-IDF + unigrams + bigrams:")
features = tfidf_vectorizer_ub.get_feature_names_out()
Top_words(features, clf)


Оценка для TF-IDF + unigrams + bigrams:

Наиболее характерные признаки для класса neautral
три звезды 3.40697
звезды 3.11118
не понравилась 2.29205
не понравились 2.01784
минусы 1.75988
ощупь но 1.75653
не очень 1.70369
супер советую 1.69923
not good 1.69393
платье велико 1.63716
не понравилось 1.5384
но плохо 1.53663
сожалению 1.53366
не понравился 1.5276
цену нормально 1.51652

--------------------

Наиболее характерные признаки для класса negative
не рекомендую 3.26344
не довольна 2.83704
не советую 2.77473
отвратительно 2.7447
деньги 2.7254
ужасно 2.71492
ужасное 2.70307
ужасная 2.69087
не 2.57466
две звезды 2.54644
говно 2.3793
не заказывайте 2.35142
ужасные 2.34836
спор 2.27681
не вернули 2.24647

--------------------

Наиболее характерные признаки для класса positive
рекомендую 4.52484
супер 4.06766
отличное 3.75875
отличная 3.73786
довольна 3.63247
отличные 3.37144
классная 3.19378
отличный 2.96124
хорошие 2.95495
хорошая 2.74478
идеально 2.73493
отлично 2.56631
хороший 2.49105

1. Какие признаки действительно связаны с тональностью текста?

Признаки в категории positive и negative очень хорошо отражают тональность текста. В категории neutral есть довольно спорные признаки, напрмер "ощупь но" и "звёзды", которые, на мой взгляд, вообще не несут особо важной информации. Есть и очень положительное "супер советую", как и очень негативное "не понравилось", которые не всегда будут соответвовать категории.    

2. Обнаружены ли признаки, которые, вероятно, отражают особенности или смещения конкретного датасета, а не тональность как таковую?

Особенности датасета могут отражать признаки:
"три звезды" - чётко говорит о категории, так как в neutral содержаться только отзывы с 3 звёздами. Аналогично с "две звезды" для класса negative




## 6. Эксперимент с предобработкой

In [50]:
import pymorphy3
import re
from sklearn.feature_extraction.text import TfidfVectorizer

token_re = re.compile(r'(?u)\b\w\w+\b')
lemmatizer = pymorphy3.MorphAnalyzer()

#Кастомная токенизация с встроенной лемматизацией
def tokenizer(text):
    words = token_re.findall(text) 
    return [lemmatizer.parse(word)[0].normal_form for word in words]

tfidf = TfidfVectorizer(tokenizer=tokenizer, token_pattern=None, ngram_range=(1, 2))

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

clf.fit(X_train_tfidf, Y_train)
pred = clf.predict(X_test_tfidf)
res(X_train_tfidf, pred)

print("\nОценка важности слов для TF-IDF + unigrams + bigrams:")
features = tfidf.get_feature_names_out()
Top_words(features, clf)



Размер словаря: 44944
Размерность матрицы признаков: (63000, 340144)
Доля ненулевых элементов в ней: 0.01
Accuracy: 0.734
Macro-F1: 0.734

Оценка важности слов для TF-IDF + unigrams + bigrams:

Наиболее характерные признаки для класса neautral
три звезда 4.37945
не понравиться 2.50994
ощупь но 1.86313
не очень 1.83629
ожидать хороший 1.78933
месяц это 1.75736
огорчить 1.72983
но плохо 1.70172
not good 1.69804
за возврат 1.66485
супер советовать 1.66096
весь рекомендовать 1.62573
ужасно вонять 1.61427
фигня выкинуть 1.59104
маломерка качество 1.58067

--------------------

Наиболее характерные признаки для класса negative
не рекомендовать 2.91714
отвратительный 2.91422
не довольный 2.78416
не советовать 2.77411
ужасный 2.63699
ужасно 2.55112
выкинуть 2.51605
говно 2.30538
не вернуть 2.29783
деньга 2.26715
не 2.21732
два звезда 2.20831
не заказывать 2.19515
невозможно 2.01376
спор 1.99822

--------------------

Наиболее характерные признаки для класса positive
отличный 5.12435
рекомендов

| Вариант                            | Размер словаря | Размерность train-матрицы | Доля ненулевых элементов | Accuracy  | Macro-F1 |
|------------------------------------|----------------|---------------------------|--------------------------|-----------|----------|
| Базовый вариант (TF-IDF + bigrams) | 44944          | (63000, 464213)           | 0.008                    | 0.735     | 0.734    |
| C лемматизацией                    | 44944          | (63000, 340144)           | 0.01                     | 0.734     | 0.734    |

В топе слов по коэфициентам модели для разных классов оказалось больше различных слов, чем было раньше. Это произошло из-за "схлопывания" одинаковых слов разной формы в одно. Я думала, это может улучшить качество классификации, но влияние, судя по метрикам, оказалось незначительным.  

Размерность матрицы получилась меньше, что в теории могло бы сократить время преобразования в новый формат тестовой выборки, но из-за предобработки вышло даже дольше. Считаю изменения не оправданными. 



## Часть 2. Диагностические эксперименты
## 8. Сравнение представлений на диагностических примерах


In [9]:
from sklearn.metrics.pairwise import cosine_similarity

A = ["Собака укусила человека.", "Человек укусил собаку."]
B = ["Мне понравился фильм.", "Мне не понравился фильм."]
C = ["Автомобиль остановился у дома.", "Машина остановилась у дома."]
D = [" Кот сидит у двери.", "Кота я увидел у двери.", "К коту подошёл ребёнок.", "С котом играет ребёнок."]

def similarity(examples):
    #Count + unigrams;
    Bow = bow_vectorizer.transform(examples)
    #TF-IDF + unigrams;
    Tfidf = tfidf_vectorizer.transform(examples)
    #TF-IDF + unigrams + bigrams.
    Tfidf_ub = tfidf_vectorizer_ub.transform(examples)
    #cosine similarity
    cs_bow = round(cosine_similarity(Bow[0], Bow[1])[0][0], 2)
    cs_tfidf = round(cosine_similarity(Tfidf[0], Tfidf[1])[0][0], 2)
    cs_tfidf_ub = round(cosine_similarity(Tfidf_ub[0], Tfidf_ub[1])[0][0], 2)
    print("\nДля предложений:")
    print(f'"{examples[0]}"')
    print(f'"{examples[1]}"')
    print("Cosine similarity в представлении BoW:", cs_bow)
    print("Cosine similarity в представлении Tf-Idf:", cs_tfidf)
    print("Cosine similarity в представлении Tf-Idf + bigrams:", cs_tfidf_ub)
    

similarity(A)
similarity(B)
similarity(C)
similarity(D[:2])
similarity(D[2:])



Для предложений:
"Собака укусила человека."
"Человек укусил собаку."
Cosine similarity в представлении BoW: 0.0
Cosine similarity в представлении Tf-Idf: 0.0
Cosine similarity в представлении Tf-Idf + bigrams: 0.0

Для предложений:
"Мне понравился фильм."
"Мне не понравился фильм."
Cosine similarity в представлении BoW: 0.87
Cosine similarity в представлении Tf-Idf: 0.99
Cosine similarity в представлении Tf-Idf + bigrams: 0.71

Для предложений:
"Автомобиль остановился у дома."
"Машина остановилась у дома."
Cosine similarity в представлении BoW: 0.5
Cosine similarity в представлении Tf-Idf: 0.21
Cosine similarity в представлении Tf-Idf + bigrams: 0.21

Для предложений:
" Кот сидит у двери."
"Кота я увидел у двери."
Cosine similarity в представлении BoW: 0.33
Cosine similarity в представлении Tf-Idf: 0.32
Cosine similarity в представлении Tf-Idf + bigrams: 0.32

Для предложений:
"К коту подошёл ребёнок."
"С котом играет ребёнок."
Cosine similarity в представлении BoW: 0.33
Cosine simila

In [16]:
#Проверка, какие слова из предложений есть в словаре
def is_in_vocabulary(examples, vectorizer):
    matrix = vectorizer.transform(examples)
    
    # Декодируем векторы обратно в слова из словаря
    words_in_doc1 = vectorizer.inverse_transform(matrix[0])[0]
    words_in_doc2 = vectorizer.inverse_transform(matrix[1])[0]
    
    print(f'Слова из словаря для 1-го текста: {words_in_doc1}')
    print(f'Слова из словаря для 2-го текста: {words_in_doc2}')

print("BoW для группы А:")
is_in_vocabulary(A, bow_vectorizer)
print("\nTf-IDF для группы А:")
is_in_vocabulary(A, tfidf_vectorizer)
print("\nTF-IDF + bigrams для группы А:")
is_in_vocabulary(A, tfidf_vectorizer_ub)



BoW для группы А:
Слова из словаря для 1-го текста: ['собака' 'человека']
Слова из словаря для 2-го текста: ['собаку' 'человек']

Tf-IDF для группы А:
Слова из словаря для 1-го текста: ['собака' 'человека']
Слова из словаря для 2-го текста: ['собаку' 'человек']

TF-IDF + bigrams для группы А:
Слова из словаря для 1-го текста: ['собака' 'человека']
Слова из словаря для 2-го текста: ['собаку' 'человек']


| Диагностический тест | Сравниваемые тексты | Count: cosine | TF-IDF: cosine  | TF-IDF + bigrams: cosine |
|----------------------|---------------------|---------------|-----------------|--------------------------|
| Порядок слов         | A1 — A2             |  0.0          | 0.0             | 0.0                      |   
| Отрицание            | B1 — B2             |  0.87         | 0.99            | 0.71                     |
| Лексическая замена   | C1 — C2             |  0.5          | 0.21            | 0.21                     |
| Словоформы           | D1 - D2             | 0.33          | 0.32            | 0.32                     |
| Словоформы           | D3 - D4             | 0.33          | 0.3             | 0.3                      |

Для примеров в первой группе значение сходства вышло равным 0 для всех представлений, так как у предложений не совпадает ни одно слово (из-за разных словоформ), а слово "укусил" вообще не учитываеться из-за отсутсвия в словаре (не было примеров с ним в обучающей выборке)

## 9. Какие признаки изменились?


In [ ]:
#Покажите программно и проанализируйте:
#- что произошло с признаками при перестановке слов;
#- что произошло с признаком `не`;
#- как представлены слова `автомобиль` и `машина`;
#- как представлены формы `кот`, `кота`, `коту`, `котом`;
#- какие дополнительные признаки появились при использовании биграмм.

import numpy as np

def presentation(examples, vectorizer):
    matrix = vectorizer.transform(examples)
    
    feature_names = np.array(vectorizer.get_feature_names_out())
    
    # Индексы ненулевых элементов для первого и второго предложения
    idx_doc1 = matrix[0].nonzero()[1]
    idx_doc2 = matrix[1].nonzero()[1]
    
    # Объединяем индексы, чтобы получить только слова из этих двух предложений
    unique_indices = list(set(idx_doc1) | set(idx_doc2))
  
    if not unique_indices:
        print("\nНи одно слово из этих предложений не найдено в словаре векторизатора.")
        return
        
    # Вырезаем подматрицу только с нужными колонками
    submatrix = matrix[:, unique_indices].toarray()
    
    # Названия колонок именно для этих индексов
    relevant_features = feature_names[unique_indices]
    
    df = pd.DataFrame(
        submatrix, 
        columns=relevant_features, 
        index=["Предложение 1 |", "Предложение 2 |"]
    )

    print("Матрица представлений (сравнение весов):")
    # Округлим для красивого вывода
    print(df.round(2)) 

print("Сравнение представлений группы А")
print(f'1: "{A[0]}"')
print(f'2: "{A[1]}"')
print("\nBoW:")
presentation(A, bow_vectorizer)
print("\nTF-IDF:")
presentation(A, tfidf_vectorizer)
print("\nTF-IDF + bigrams:")
presentation(A, tfidf_vectorizer_ub)




Сравнение представлений группы А
1: "Собака укусила человека."
2: "Человек укусил собаку."

BoW:
Матрица представлений (сравнение весов):
                 человек  собаку  собака  человека
Предложение 1 |        0       0       1         1
Предложение 2 |        1       1       0         0

TF-IDF:
Матрица представлений (сравнение весов):
                 человек  собаку  собака  человека
Предложение 1 |     0.00    0.00    0.81      0.59
Предложение 2 |     0.58    0.81    0.00      0.00

TF-IDF + bigrams:
Матрица представлений (сравнение весов):
                 собака  человека  собаку  человек
Предложение 1 |    0.81      0.59    0.00     0.00
Предложение 2 |    0.00      0.00    0.81     0.58


Для группы А (порядок слов) представления различают тексты, но не кодируют интересующее нас отношение между ними; Более того, если бы тексты были подвергнуты лемматизации, они бы не различались в этом представлении, так как сейчас различия идут только за счёт словоформы

In [26]:
print("Сравнение представлений группы B")
print(f'1: "{B[0]}"')
print(f'2: "{B[1]}"')
print("\nBoW:")
presentation(B, bow_vectorizer)
print("\nTF-IDF:")
presentation(B, tfidf_vectorizer)
print("\nTF-IDF + bigrams:")
presentation(B, tfidf_vectorizer_ub)

Сравнение представлений группы B
1: "Мне понравился фильм."
2: "Мне не понравился фильм."

BoW:
Матрица представлений (сравнение весов):
                 мне  понравился  фильм  не
Предложение 1 |    1           1      1   0
Предложение 2 |    1           1      1   1

TF-IDF:
Матрица представлений (сравнение весов):
                  мне  понравился  фильм    не
Предложение 1 |  0.28        0.43   0.86  0.00
Предложение 2 |  0.28        0.43   0.85  0.12

TF-IDF + bigrams:
Матрица представлений (сравнение весов):
                 понравился  мне не   не  мне понравился   мне  фильм  \
Предложение 1 |        0.37    0.00  0.0            0.53  0.24   0.73   
Предложение 2 |        0.36    0.34  0.1            0.00  0.23   0.72   

                 не понравился  
Предложение 1 |           0.00  
Предложение 2 |           0.42  


В группе B видно, что в представлениях TF-IDF и BoW векторы отличаються совсем немного и их противоположное значение не кодируеться. Лучше всего различия улавливаються с помощью биграмм, так как появляються новые отличные токены. Но они всё равно плохо отражают противоположное значение предложений, так как веса у "не", "мне не", "не понравился" недостаточно высокие. 

In [28]:
print("Сравнение представлений группы С")
print(f'1: "{C[0]}"')
print(f'2: "{C[1]}"')
print("\nBoW:")
presentation(C, bow_vectorizer)
print("\nTF-IDF:")
presentation(C, tfidf_vectorizer)
print("\nTF-IDF + bigrams:")
presentation(C, tfidf_vectorizer_ub)

Сравнение представлений группы С
1: "Автомобиль остановился у дома."
2: "Машина остановилась у дома."

BoW:
Матрица представлений (сравнение весов):
                 остановилась  остановился  дома
Предложение 1 |             0            1     1
Предложение 2 |             1            0     1

TF-IDF:
Матрица представлений (сравнение весов):
                 остановилась  остановился  дома
Предложение 1 |          0.00         0.89  0.46
Предложение 2 |          0.89         0.00  0.45

TF-IDF + bigrams:
Матрица представлений (сравнение весов):
                 остановилась  остановился  дома
Предложение 1 |          0.00         0.89  0.46
Предложение 2 |          0.89         0.00  0.45


Для группы C слова "автомобиль" и "машина" вообще никак не представленны в словарях, из-за тематики обучающего датасета. Но даже если бы они были представленны, их синонимичность никак не закодировалась бы в этих представлениях. 

In [30]:
print("Сравнение представлений группы D")
print(f'1: "{D[0]}"')
print(f'2: "{D[1]}"')
print("\nBoW:")
presentation(D[:2], bow_vectorizer)
print("\nTF-IDF:")
presentation(D[:2], tfidf_vectorizer)
print("\nTF-IDF + bigrams:")
presentation(D[:2], tfidf_vectorizer_ub)

Сравнение представлений группы D
1: " Кот сидит у двери."
2: "Кота я увидел у двери."

BoW:
Матрица представлений (сравнение весов):
                 увидел  сидит  двери  кот  кота
Предложение 1 |       0      1      1    1     0
Предложение 2 |       1      0      1    0     1

TF-IDF:
Матрица представлений (сравнение весов):
                 увидел  сидит  двери  кот  кота
Предложение 1 |    0.00   0.37   0.62  0.7  0.00
Предложение 2 |    0.62   0.00   0.52  0.0  0.58

TF-IDF + bigrams:
Матрица представлений (сравнение весов):
                 двери  кот  увидел  кота  сидит
Предложение 1 |   0.62  0.7    0.00  0.00   0.37
Предложение 2 |   0.52  0.0    0.62  0.58   0.00


Представления не различают разные формы слова. 

## 10. Влияние представления на предсказание модели

In [35]:
#Для каждого предложения сохраните:
#- используемый вариант представления;
#- предсказанный класс;
#- значение predict_proba, decision_function или другой доступной меры уверенности модели
from sklearn.svm import LinearSVC

#Натринеруем классифиакторы отдельно для каждого формата данных (до этого просто переучивался один классификатор)
bow_clf = LinearSVC(random_state=42)
bow_clf.fit(X_train_bow, Y_train)

tfidf_clf = LinearSVC(random_state=42)
tfidf_clf.fit(tfidf_vectorizer.fit_transform(X_train), Y_train)

tfidf_ub_clf = LinearSVC(random_state=42)
tfidf_ub_clf.fit(tfidf_vectorizer_ub.fit_transform(X_train), Y_train)


,penalty,'l2'
,loss,'squared_hinge'
,dual,'auto'
,tol,0.0001
,C,1.0
,multi_class,'ovr'
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,verbose,0
,random_state,42


In [ ]:
#Считает и выводит предсказанные классы и уверенность модели в них 
def compute_class(examples, vectorizer, clf):
    matrix = vectorizer.transform(examples)
    preds = clf.predict(matrix)
    scores = clf.decision_function(matrix)
    
    for i in range(len(preds)):
        print(f"{i}. Предсказание: {preds[i]} Уверенность: {max(abs(scores[i])):.2f}")

print("Предложения:")
groups = [A, B, C, D]
examples =[]
i = 0
for group in groups:
    for sentence in group:
        print(f"{i}: {sentence}")
        examples.append(sentence)
        i += 1

print("\nРезультаты для BoW:")
compute_class(examples, bow_vectorizer, bow_clf)
print("\nРезультаты для TF-IDF:")
compute_class(examples, tfidf_vectorizer, tfidf_clf)
print("\nРезультаты для TF-IDF + bigrams:")
compute_class(examples, tfidf_vectorizer_ub, tfidf_ub_clf)



Предложения:
0: Собака укусила человека.
1: Человек укусил собаку.
2: Мне понравился фильм.
3: Мне не понравился фильм.
4: Автомобиль остановился у дома.
5: Машина остановилась у дома.
6:  Кот сидит у двери.
7: Кота я увидел у двери.
8: К коту подошёл ребёнок.
9: С котом играет ребёнок.

Результаты для BoW:
0. Предсказание: negative Уверенность: 0.96
1. Предсказание: neautral Уверенность: 0.90
2. Предсказание: neautral Уверенность: 0.84
3. Предсказание: neautral Уверенность: 0.54
4. Предсказание: neautral Уверенность: 0.41
5. Предсказание: positive Уверенность: 1.56
6. Предсказание: neautral Уверенность: 1.05
7. Предсказание: positive Уверенность: 1.24
8. Предсказание: positive Уверенность: 1.24
9. Предсказание: neautral Уверенность: 2.55

Результаты для TF-IDF:
0. Предсказание: negative Уверенность: 1.00
1. Предсказание: neautral Уверенность: 0.77
2. Предсказание: neautral Уверенность: 0.88
3. Предсказание: neautral Уверенность: 0.62
4. Предсказание: neautral Уверенность: 0.59
5. Пред

| Предложение                   | Метка Bow | Уверенность BoW | Метка TF-IDF | Уверенность TF-IDF  | Метка TF-IDF + bigrms | Уверенность TF-IDF + bigrams |
|-------------------------------|-----------|-----------------|--------------|---------------------|-----------------------|------------------------------|
| Собака укусила человека.      | negative  |      0.96       | negative     |  1.00               |    negative           | 0.78
| Человек укусил собаку.        | neautral  |      0.90       | neautral     |  0.77               |    neautral           | 0.59
| Мне понравился фильм.         | neautral  |      0.84       | neautral     |  0.88               |    positive           | 1.16
| Мне не понравился фильм.      | neautral  |      0.54       | neautral     |  0.62               |    neautral           | 1.46
| Автомобиль остановился у дома.| neautral  |      0.41       | neautral     |  0.59               |    neautral           | 0.48
| Машина остановилась у дома.   | positive  |      1.56       | neautral     |  0.91               |    neautral           | 0.47
| Кот сидит у двери.            | neautral  |      1.05       | neautral     |  1.01               |    neautral           | 0.75
| Кота я увидел у двери.        | positive  |      1.24       | negative     |  1.12               |    negative           | 0.69
| К коту подошёл ребёнок.       | positive  |      1.24       | positive     |  1.01               |    positive           | 1.05
| С котом играет ребёнок.       | neautral  |      2.55       | neautral     |  1.70               |    neautral           | 0.92

Анализ на примерах:

1. "Мне понравился фильм" - "Мне не понравился фильм"

В данном случае смысл предлажения изменился кардинально, а его векторное представление, как уже было выяснено ранне, почти не изменилось, при переходе от BoW к TF-IDF (добавился токен "не"). И предсказания модели для разных представлений также остались одинаковыми (neutral). Однако, изменилась уверенность модели в ответе. Для предложения "Мне не понравился фильм", она стала меньше, что значит, что частица "не" добавила некий вклад в окончательное решение, но не настолько большой, чтобы его изменить.

2. "Автомобиль остановился у дома" - "Машина остановилась у дома"

Здесть смысл предложений остался одинковым, но их математическое представление изменилось из-за разных словоформ. Из-за этого в представлении BoW у них оказались разные метки. Tf-IDF, напротив, смог сохранить метки одинаковыми. Т.е. в данном случае изменение представление помогло сделать результат более стабильным. 

3. "Мне понравился фильм"

При переходе представления от обычного TF-IDF к TF-IDF с биграммами количество получаемой моделью информации возрасло и метка изменилась от "neutral" к гораздо более уверенному "positive". Биграммы помагли захватить нужный контекст и правильно его классифицировать.  


## Часть 3. Собственные диагностические тесты

## 11. Разработка собственных тестов


In [48]:
#- устойчивые словосочетания;
#- усилители: например, `хороший` / `очень хороший`;
# - опечатки;

#Тексты сформулированны с учётом тематики обучающей выборки (отзывы на женские товары/аксесуары)
E = ["Продавец тянул кота за хвост с отправкой заказа", "Продавец медлил с отправкой заказа"]
F = ["Красивая, качественная ткань", "Очень красивая, качественная ткань"]
J = ["Леггинсы не подошли", "Легенсы не подошли"]

print("Сравнение представлений группы E")
print(f'0: "{E[0]}"')
print(f'1: "{E[1]}"')
print("\nBoW:")
presentation(E, bow_vectorizer)
print("\nTF-IDF:")
presentation(E, tfidf_vectorizer)

similarity(E)

print("\nПредсказания для BoW:")
compute_class(E, bow_vectorizer, bow_clf)
print("\nПредсказания для TF-IDF:")
compute_class(E, tfidf_vectorizer, tfidf_clf)


Сравнение представлений группы E
0: "Продавец тянул кота за хвост с отправкой заказа"
1: "Продавец медлил с отправкой заказа"

BoW:
Матрица представлений (сравнение весов):
                 тянул  заказа  за  хвост  кота  продавец  отправкой
Предложение 1 |      1       1   1      1     1         1          1
Предложение 2 |      0       1   0      0     0         1          1

TF-IDF:
Матрица представлений (сравнение весов):
                 тянул  заказа    за  хвост  кота  продавец  отправкой
Предложение 1 |   0.43    0.31  0.17   0.48  0.51      0.17       0.42
Предложение 2 |   0.00    0.56  0.00   0.00  0.00      0.31       0.77

Для предложений:
"Продавец тянул кота за хвост с отправкой заказа"
"Продавец медлил с отправкой заказа"
Cosine similarity в представлении BoW: 0.65
Cosine similarity в представлении Tf-Idf: 0.55
Cosine similarity в представлении Tf-Idf + bigrams: 0.57

Предсказания для BoW:
0. Предсказание: neautral Уверенность: 0.98
1. Предсказание: neautral Уверенность

В этом эксперементе проверяласось влияние устойчивых словосочетаний. Хоть математические представления предложений и изменились, Bow отнёс их к одной категории, т.к. все слова нейтральные. TF-IDF же для первого предложения определил метку negative. Это следствие того, что слова «хвост» (вес 0.48) и «кота» (вес 0.51) получили огромный вес. По всей видимости, в обучающем датасете эти слова (или похожие контексты с ними) встречались преимущественно в негативных отзывах. Предложение же без фразеологизма было оценено как нейтральное, скорее всего из-за отсутсвия в словере слов "медлил".

In [49]:
print("Сравнение для группы F")
print(f'0: "{F[0]}"')
print(f'1: "{F[1]}"')
print("\nBoW:")
presentation(F, bow_vectorizer)
print("\nTF-IDF:")
presentation(F, tfidf_vectorizer)

similarity(F)

print("\nПредсказания для BoW:")
compute_class(F, bow_vectorizer, bow_clf)
print("\nПредсказания для TF-IDF:")
compute_class(F, tfidf_vectorizer, tfidf_clf)

Сравнение для группы F
0: "Красивая, качественная ткань"
1: "Очень красивая, качественная ткань"

BoW:
Матрица представлений (сравнение весов):
                 ткань  очень  качественная  красивая
Предложение 1 |      1      0             1         1
Предложение 2 |      1      1             1         1

TF-IDF:
Матрица представлений (сравнение весов):
                 ткань  очень  качественная  красивая
Предложение 1 |   0.38   0.00          0.69      0.62
Предложение 2 |   0.37   0.24          0.67      0.60

Для предложений:
"Красивая, качественная ткань"
"Очень красивая, качественная ткань"
Cosine similarity в представлении BoW: 0.87
Cosine similarity в представлении Tf-Idf: 0.97
Cosine similarity в представлении Tf-Idf + bigrams: 0.91

Предсказания для BoW:
0. Предсказание: positive Уверенность: 1.32
1. Предсказание: positive Уверенность: 1.30

Предсказания для TF-IDF:
0. Предсказание: positive Уверенность: 1.77
1. Предсказание: positive Уверенность: 1.76


В тесте с усиливающим словом представление различает тексты, но это практически не отражаеться на результатах классификации. Вместо того, чтобы усилить положительное влияние, уверенность модели для обоих представлений упала. Это связано с тем, что слово "очень" могло одинаково часто встречаться как в положительном, так и в негативном контексте и в данном случае скорее являеться "шумом". Тем не менее, TF-IDF справился немного лучше: за счет L2-нормализации и штрафа за общую частоту слова (IDF) вес "шумового" токена "очень" был минимизирован, что позволило сохранить косинусное сходство между векторами на высоком уровне (0.97) и не дать уверенности модели упасть так же сильно, как в BoW

In [50]:
print("Сравнение для группы J")
print(f'0: "{J[0]}"')
print(f'1: "{J[1]}"')
print("\nBoW:")
presentation(J, bow_vectorizer)
print("\nTF-IDF:")
presentation(J, tfidf_vectorizer)

similarity(J)

print("\nПредсказания для BoW:")
compute_class(J, bow_vectorizer, bow_clf)
print("\nПредсказания для TF-IDF:")
compute_class(J, tfidf_vectorizer, tfidf_clf)

Сравнение для группы J
0: "Леггинсы не подошли"
1: "Легенсы не подошли"

BoW:
Матрица представлений (сравнение весов):
                 леггинсы  не  подошли  легенсы
Предложение 1 |         1   1        1        0
Предложение 2 |         0   1        1        1

TF-IDF:
Матрица представлений (сравнение весов):
                 леггинсы    не  подошли  легенсы
Предложение 1 |      0.71  0.18     0.68     0.00
Предложение 2 |      0.00  0.15     0.57     0.81

Для предложений:
"Леггинсы не подошли"
"Легенсы не подошли"
Cosine similarity в представлении BoW: 0.67
Cosine similarity в представлении Tf-Idf: 0.41
Cosine similarity в представлении Tf-Idf + bigrams: 0.39

Предсказания для BoW:
0. Предсказание: neautral Уверенность: 0.55
1. Предсказание: neautral Уверенность: 0.82

Предсказания для TF-IDF:
0. Предсказание: neautral Уверенность: 0.62
1. Предсказание: neautral Уверенность: 0.92


В тексте с опечатками представления различают, но не кодирует отношения между словами. В TF-IDF вес для слова с ошибкой оказался больше, скорее всего за счёт того, что оно реже употреблялось в датасете (Т.к. благодаря IDF частые слова получают меньший вес). Из-за этого же уверенность классификации для этого предложения оказалась больше, потому что правильное слово могло часто встречаться в разных категориях и за счёт этого вносить небольшой шум.   

In [55]:
#Посчитаем метрику для сравнения представлений
#Изменение косинусного сходства при переходе от BoW к TF-IDF
cos_bow = [0.65, 0.87, 0.67]
cos_tfidf = [0.55, 0.97, 0.41]

print("Разность для группы с устойчивым словосочетанием:")
print(round(cos_tfidf[0] - cos_bow[0], 2))

print("\nРазность для группы с усилением:")
print(round(cos_tfidf[1] - cos_bow[1], 2))

print("\nРазность для группы с опечатками:")
print(round(cos_tfidf[2] - cos_bow[2], 2))



Разность для группы с устойчивым словосочетанием:
-0.1

Разность для группы с усилением:
0.1

Разность для группы с опечатками:
-0.26


Тест                | Исследуемое свойство | Сравниваемые представления | Основной измеренный результат | Вывод 
--------------------|----------------------|----------------------------|-------------------------------|-------
Собственный тест 1  | устойчивые выражения | Bow, TF-IDF                | -0.1                          | TF-IDF отдаляет друг от друга предложения с идиомами и без них, так как редкие слова получают больший вес и действуют как шум
Собственный тест 2  | усиления             | Bow, TF-IDF                |  0.1                          | Усилители, напротив, сближают векторы предложений в TF-IDF, так как из-за их частой встречаемости они получают меньший вес
Собственный тест 3  | опечатки             | Bow, TF-IDF                | -0.26                         | TF-IDF хуже справляеться с опечатками, так как из-за свое  редкости они перетягивают на себя больший вес и вектор отдаляеться от вектора с правильным вариантом


## Часть 4. Итоговый анализ

## 12. Итоговые выводы


1. Как выбор между Count и TF-IDF повлиял на представление и качество классификации?

TF-IDF стал считать не только как часто слово встречаеться в конкретном документе (отзыве), но и то, насколько оно распространено относительно других документов корпуса. Т.е. уже учитывается больше информации и у слов появляеться разный вклад в окончательную формулу. Слова которые встречаються часто получают меньший вес, так как хуже различают разные тексты, в то время как редки слова дают больший вклад. Это позволило немного увеличить качесвто классификации  (Macro-F1 для BoW было 0.709, для TF-IDF стало 0.721).

2. Что дало добавление биграмм и какой ценой?

Добавление биграмм позволило учитывать порядок слов в тексе, за счёт добавления словосочетаний. Из-за этого вырос словарь векторизатора и увеличилась разреженность матрицы представлений.

3. Какие ограничения разреженных представлений наиболее явно проявились в диагностических
тестах?

Отсутвие влияния контекста на представления. Посокольку BoW и TF-IDF основаны на подсчёте слов, они совершенно слепы к контексту, перестановке слов, не улавливают синонимов, отрицаний или усилений. Это частично решается добавлением биграмм, но из-за этого также растёт разреженность и иногда дополнительная информация может только размывать смысл предложения.  

4. Какие ошибки классификатора удалось связать со свойствами representation?

Первое: возросшая уверенность классификатора в ответе, при появлении опечаток, в TF-IDF. Обусловлено тем, что в TF-IDF более редкие слова (с опечатками) получают больший вес и вносят больший вклад в выбор класса. 

Второе: размытие смысла предложения в TF-IDF с биграммами. Видно на отзыве "качество ужасное и не совсем соответствует описанию.", который в обычном TF-IDF получил правильную метку negative, а с биграммами оказался neutral. С биграммами негативными остальись только "ужасное" и "качество ужасное", чей вес уменьшился из-за добавления большего количества нейтральных словосочетаний.  

Третье: Минимальное изменение представлений при добавлении отрицания в BoW и TF-IDF (меняеться одна координата в векторе).

5. Какие ошибки нельзя объяснить только выбором representation?

Первое: Удаление знаков препинания, которые сильно влияют на смысл предложения. Например, в предложении "товар пришел с браком!!! дырка на товаре!!!" восклицательные знаки сильно усиливают отрицательное значение, но в представлении текста никак не учавствуют. Это изъян самого токенизатора, а не конкретного представления. 

Второе: Ошибки классификации из-за отсутвия некторох слов в датасете.  

6. Что показал эксперимент с предобработкой?

Приведение набора данных в представление с лемматизацией заняло больше времени, а прероста в качестве классификации не дало. Но, при этом уменьшилась разреженность словаря и его объём. В эксперементах, где проверялись словоформы вариант с лемматизацией точно справился бы лучше, так что в некторых задачах её использование будет актуальным.  

7. Какой вариант системы вы выбрали бы в качестве baseline для дальнейших экспериментов и
почему?

Я бы выбрала вариант TF-IDF + unigrams + LinearCSV, так как он показал неплохое качество классификации, его легко интерпретировать и словарь так сильно не разрастаеться на крупных наборах данных, как TF-IDF c биграммами. 

Ещё неплохой вариант TF-IDF + unigrams + bigrams + LinearCSV, так как на нём получаеться лучшее качество классификации, но тут придёться дополнительно настраивать классификатоора и векторизатор, чтобы уменьшить негативное влияние большой разреженности матрицы и размерности словаря. Например, ввести макисальное и минимальное ограничение частоты слов на добавление в словарь (параметры min_df и max_df в векторизаторе), сделать лемматизацию или добавить регуляризацию весов. Но это уже более сложная система, а для basline, чем проще - тем лучше.   
